CODE

MODEL ARCHITECTURE : https://www.kaggle.com/code/madz2000/cnn-using-keras-100-accuracy

WANDB DOCUMENTATION KERAS INTEGRATION : https://docs.wandb.ai/models/integrations/keras

In [15]:
import os

import kagglehub
import numpy as np
import pandas as pd
import string
import json
from sklearn.preprocessing import LabelBinarizer
from PIL import Image

import wandb
from wandb.integration.keras import WandbMetricsLogger

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, BatchNormalization, Dropout, Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, ModelCheckpoint

In [16]:
wandb.login()

True

In [17]:
# DATA LOADING AND PREPARATION

ENVIRONMENT = "local"

if ENVIRONMENT == "colab":
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("Google Drive mounted.")
    except Exception as e:
        print(f"Drive already mounted or error mounting: {e}")

    CUSTOM_TEST_DATA_INPUT_DIR = "/content/drive/MyDrive/ColabTestData"
    PROCESSED_TEST_DATA_OUTPUT_DIR = "/content/test_images_processed"

elif ENVIRONMENT == "local":
    CUSTOM_TEST_DATA_INPUT_DIR = "./ColabTestData"
    PROCESSED_TEST_DATA_OUTPUT_DIR = "./test_images_processed"
    print(f"Running in 'local' mode.")
    print(f"Expecting custom test data in: {os.path.abspath(CUSTOM_TEST_DATA_INPUT_DIR)}")
else:
    raise ValueError("ENVIRONMENT must be set to 'colab' or 'local'")

LOG_DIR = "./logs/full_run"
CHECKPOINT_PATH = "./checkpoints/best_model.weights.h5"
SANITY_LOG_DIR = "./logs/sanity_check"
HPARAMS_PATH = os.path.join(LOG_DIR, "hparams.json")

os.makedirs(PROCESSED_TEST_DATA_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
os.makedirs(SANITY_LOG_DIR, exist_ok=True)

print("\nRunning Data Processing")
path = kagglehub.dataset_download("datamunge/sign-language-mnist")

all_letters = string.ascii_uppercase
label_to_letter = {}
letter_to_label = {}

for i, char in enumerate(all_letters):
    if char not in ['J', 'Z']:
        dataset_label = i
        label_to_letter[dataset_label] = char
        letter_to_label[char] = dataset_label

input_folder = CUSTOM_TEST_DATA_INPUT_DIR
output_folder = PROCESSED_TEST_DATA_OUTPUT_DIR
csv_data = []

if not os.path.exists(input_folder) or not os.listdir(input_folder):
    print(f"Warning: Input folder {input_folder} is empty or not found.")
else:
    for filename in os.listdir(input_folder):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            try:
                img_path = os.path.join(input_folder, filename)
                img = Image.open(img_path).convert('L') # grayscale
                img = img.resize((28, 28)) # resize to 28x28
                img.save(os.path.join(output_folder, filename))

                label_char = filename.split('.')[0][-1].upper()
                if label_char in letter_to_label:
                    label = letter_to_label[label_char]
                    img_array = np.array(img).flatten()
                    row_data = [label] + img_array.tolist()
                    csv_data.append(row_data)
                else:
                    print(f"Skipping {filename}: Label '{label_char}' not in mapping.")
            except Exception as e:
                print(f"Error processing {filename}: {e}")

output_filename = "test_images.csv"
headers = ['label'] + [f'pixel{i+1}' for i in range(28*28)]
csv_path = os.path.join(output_folder, output_filename)

if csv_data:
    df = pd.DataFrame(csv_data, columns=headers)
    df.to_csv(csv_path, index=False)
    print(f"Successfully processed {len(csv_data)} images and saved custom dataset to: {csv_path}")
else:
    print("No custom images were processed.")
    df = pd.DataFrame(columns=headers)
    df.to_csv(csv_path, index=False)
    print(f"Created empty test_images.csv at: {csv_path}")

train_df = pd.read_csv(os.path.join(path, "sign_mnist_train/sign_mnist_train.csv"))
val_df = pd.read_csv(os.path.join(path, "sign_mnist_test/sign_mnist_test.csv"))
test_df = pd.read_csv(csv_path)

print(f"\nLoaded datasets:")
print(f"  Training data shape:    {train_df.shape}")
print(f"  Validation data shape: {val_df.shape}")
print(f"  Custom Test data shape:  {test_df.shape}")



print("\n Preprocessing Data for Model: ")

y_train = train_df['label']
x_train = train_df.drop('label', axis=1)
y_val = val_df['label']
x_val = val_df.drop('label', axis=1)
y_test = test_df['label']
x_test = test_df.drop('label', axis=1)


x_train = x_train.values.reshape(-1, 28, 28, 1)
x_val = x_val.values.reshape(-1, 28, 28, 1)
print(f"Reshaped x_train: {x_train.shape}")
print(f"Reshaped x_val: {x_val.shape}")


lb = LabelBinarizer()
y_train_bin = lb.fit_transform(y_train)
y_val_bin = lb.transform(y_val)
num_classes = y_train_bin.shape[1]
print(f"One-hot encoded labels. Found {num_classes} classes.")

if not test_df.empty:
    x_test = x_test.values.reshape(-1, 28, 28, 1)
    y_test_bin = lb.transform(y_test)
    print(f"Reshaped x_test: {x_test.shape}")
else:
    x_test = np.array([])
    y_test_bin = np.array([])
    print("Custom test set is empty. Skipping its preprocessing.")


config = {
    "learning_rate": 0.001,
    "optimizer": "adam",
    "loss_function": "categorical_crossentropy",
    "batch_size": 128,
    "epochs": 20,
    "rotation_range": 10,
    "zoom_range": 0.1,
    "width_shift_range": 0.1,
    "height_shift_range": 0.1
}

with open(HPARAMS_PATH, 'w') as f:
    json.dump(config, f, indent=4)
print(f"Hyperparameters saved to {HPARAMS_PATH}")

Running in 'local' mode.
Expecting custom test data in: /home/polemaster/uni/semester7/dnn/phase2/ColabTestData

Running Data Processing
Successfully processed 96 images and saved custom dataset to: ./test_images_processed/test_images.csv

Loaded datasets:
  Training data shape:    (27455, 785)
  Validation data shape: (7172, 785)
  Custom Test data shape:  (96, 785)

 Preprocessing Data for Model: 
Reshaped x_train: (27455, 28, 28, 1)
Reshaped x_val: (7172, 28, 28, 1)
One-hot encoded labels. Found 24 classes.
Reshaped x_test: (96, 28, 28, 1)
Hyperparameters saved to ./logs/full_run/hparams.json


In [18]:
# DATA AUGMENTATION

print("\nSetting up Data Augmentation")

datagen = ImageDataGenerator(
        rescale=1./255,
        featurewise_center=False,
        samplewise_center=False,
        featurewise_std_normalization=False,
        samplewise_std_normalization=False,
        zca_whitening=False,
        rotation_range=config['rotation_range'],
        zoom_range = config['zoom_range'],
        width_shift_range=config['width_shift_range'],
        height_shift_range=config['height_shift_range'],
        horizontal_flip=False,
        vertical_flip=False)

val_datagen = ImageDataGenerator(rescale=1./255)

datagen.fit(x_train)

# MODEL DEFINITION

def build_model(num_classes):
    model = Sequential([
        Conv2D(75 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu' , input_shape = (28,28,1)),
        BatchNormalization(),
        MaxPool2D((2,2) , strides = 2 , padding = 'same'),
        Conv2D(50 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu'),
        Dropout(0.2),
        BatchNormalization(),
        MaxPool2D((2,2) , strides = 2 , padding = 'same'),
        Conv2D(25 , (3,3) , strides = 1 , padding = 'same' , activation = 'relu'),
        BatchNormalization(),
        MaxPool2D((2,2) , strides = 2 , padding = 'same'),
        Flatten(),
        Dense(units = 512 , activation = 'relu'),
        Dropout(0.3),
        Dense(units = num_classes , activation = 'softmax') # Use num_classes
    ])

    model.compile(
        optimizer = config['optimizer'],
        loss = config['loss_function'],
        metrics = ['accuracy']
    )
    return model


Setting up Data Augmentation


In [19]:
# SANITY CHECK

print("\nSANITY CHECK")

run = wandb.init(
    project="sign-language-mnist",
    name="sanity_check",
    config=config,
    reinit=True,
    dir=SANITY_LOG_DIR
)

(x_batch, y_batch) = next(datagen.flow(x_train, y_train_bin, batch_size=config['batch_size']))

sanity_model = build_model(num_classes)

sanity_model.fit(
    x_batch,
    y_batch,
    epochs=50,
    callbacks=[WandbMetricsLogger()],
    verbose=0
)

run.finish()


SANITY CHECK


/home/polemaster/uni/semester7/dnn/phase2/venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


epoch/accuracy,▁▂▃▅▆▇▇█████████████████████████████████
epoch/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,1
epoch/epoch,49
epoch/learning_rate,0.001
epoch/loss,0.0084


In [20]:
# FULL TRAINING RUN

print("\nLAUNCHING FULL TRAINING RUN ")

run = wandb.init(
    project="sign-language-mnist",
    name="full_run",
    config=config,
    reinit=True,
    dir=LOG_DIR
)

model = build_model(num_classes)
model.summary()


learning_rate_reduction = ReduceLROnPlateau(
    monitor='val_accuracy',
    patience=2,
    verbose=1,
    factor=0.5,
    min_lr=0.00001
)


model_checkpoint = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)


wandb_callback = WandbMetricsLogger()


callbacks_list = [
    learning_rate_reduction,
    model_checkpoint,
    wandb_callback
]

history = model.fit(
    datagen.flow(x_train, y_train_bin, batch_size=config['batch_size']),
    epochs = config['epochs'],
    validation_data = val_datagen.flow(x_val, y_val_bin, batch_size=config['batch_size']),
    steps_per_epoch = len(x_train) // config['batch_size'],
    validation_steps = len(x_val) // config['batch_size'],
    callbacks = callbacks_list
)

print("Full training complete.")
run.finish()


LAUNCHING FULL TRAINING RUN 


/home/polemaster/uni/semester7/dnn/phase2/venv/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_15 (Conv2D)              │ (None, 28, 28, 75)     │           750 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 28, 28, 75)     │           300 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 14, 14, 75)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 14, 14, 50)     │        33,800 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 14, 14, 50)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 14, 14, 50)     │           200 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_16 (MaxPooling2D) │ (None, 7, 7, 50)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 7, 7, 25)       │        11,275 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 7, 7, 25)       │           100 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_17 (MaxPooling2D) │ (None, 4, 4, 25)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_5 (Flatten)             │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 512)            │       205,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 24)             │        12,312 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 264,049 (1.01 MB)

...

 Non-trainable params: 300 (1.17 KB)

Epoch 1/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.4889 - loss: 1.7156
Epoch 1: val_accuracy improved from None to 0.12333, saving model to ./checkpoints/best_model.weights.h5
214/214 ━━━━━━━━━━━━━━━━━━━━ 15s 43ms/step - accuracy: 0.6940 - loss: 0.9687 - val_accuracy: 0.1233 - val_loss: 3.5790 - learning_rate: 0.0010
Epoch 2/20
  1/214 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.9141 - loss: 0.2733

/home/polemaster/uni/semester7/dnn/phase2/venv/lib/python3.13/site-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()



Epoch 2: val_accuracy improved from 0.12333 to 0.13714, saving model to ./checkpoints/best_model.weights.h5
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9141 - loss: 0.2733 - val_accuracy: 0.1371 - val_loss: 3.5636 - learning_rate: 0.0010
Epoch 3/20
213/214 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.9227 - loss: 0.2368
Epoch 3: val_accuracy improved from 0.13714 to 0.60896, saving model to ./checkpoints/best_model.weights.h5
214/214 ━━━━━━━━━━━━━━━━━━━━ 5s 22ms/step - accuracy: 0.9364 - loss: 0.1953 - val_accuracy: 0.6090 - val_loss: 1.2591 - learning_rate: 0.0010
Epoch 4/20
  1/214 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9922 - loss: 0.0548
Epoch 4: val_accuracy did not improve from 0.60896
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.9922 - loss: 0.0548 - val_accuracy: 0.6039 - val_loss: 1.2788 - learning_rate: 0.0010
Epoch 5/20
214/214 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.9634 - loss: 0.1142
Epoch 5: val_accuracy improved from 0.60896 to 0

epoch/accuracy,▁▆▇█▇▇███▇██████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,████████▄▄▄▄▂▂▁▁▁▁▁▁
epoch/loss,█▃▂▁▂▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▁▅▅█▇██████████████
epoch/val_loss,██▃▃▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,1
epoch/epoch,19
epoch/learning_rate,6e-05
epoch/loss,0.00514
epoch/val_accuracy,0.99581


In [21]:
# FINAL EVALUATION

print("\nEvaluating Best Model")
print(f"Loading best weights from: {CHECKPOINT_PATH}")

model.load_weights(CHECKPOINT_PATH)

print("Evaluating on Validation Set (sign_mnist_test.csv):")
val_loss, val_acc = model.evaluate(val_datagen.flow(x_val, y_val_bin))
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")

if not test_df.empty:
    print("\nEvaluating on Custom Test Set (test_images.csv):")
    test_loss, test_acc = model.evaluate(val_datagen.flow(x_test, y_test_bin))
    print(f"Custom Test Loss: {test_loss:.4f}, Custom Test Accuracy: {test_acc:.4f}")
else:
    print("\nCustom test set is empty, skipping final evaluation on it.")


Evaluating Best Model
Loading best weights from: ./checkpoints/best_model.weights.h5
Evaluating on Validation Set (sign_mnist_test.csv):
225/225 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9964 - loss: 0.0072
Validation Loss: 0.0072, Validation Accuracy: 0.9964

Evaluating on Custom Test Set (test_images.csv):
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5729 - loss: 2.3572 
Custom Test Loss: 2.3572, Custom Test Accuracy: 0.5729
